In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import xarray as xr
import seaborn as sns
sns.set_theme("notebook",style="dark")

In [ ]:
import os
from pathlib import Path

from qgsw.logging.core import getLogger, setup_root_logger
from qgsw.specs import defaults


torch.backends.cudnn.deterministic = True

ROOT_PATH = Path(os.getcwd()).parent


specs=defaults.get()

logger=getLogger(__name__)
setup_root_logger(1)

In [ ]:
data_dt = 7200
dt = 7200
n_file_per_cycle = 20
n_steps_per_cycle = 240
separation = 0
comparison_interval = 1
n_cycles=1
b = 4

In [ ]:
from qgsw.eNATL60 import seasons
from qgsw.eNATL60.loading import retrieve_dates, sort_files_by_dates
from qgsw.utils.storage import get_path_from_env


data_folder = get_path_from_env(key="eNATL60_FOLDER")
files = list((data_folder / "MEANDERS" / "gridT").glob("*.nc"))

files = sort_files_by_dates(*files)

season_dict = {
    "summer": seasons.SUMMER,
    "autumn": seasons.AUTUMN,
    "winter": seasons.WINTER,
    "spring": seasons.SPRING,
}

In [ ]:
### Load only one file to access grid informations

from qgsw.spatial.core.grid import Grid2D
from qgsw.physics.constants import EARTH_ANGULAR_ROTATION
from qgsw.eNATL60.interpolation import build_regridder, compute_lonlat_from_regular_xy_grid, lonlat_to_xy
from qgsw.eNATL60.loading import load_datasets
from qgsw.eNATL60.var_keys import LATITUDE, LONGITUDE
from qgsw.physics.constants import EARTH_RADIUS
from qgsw.physics.coriolis.beta_plane import BetaPlane
from qgsw.scripts.eNATL60 import format_ds
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate


ds = load_datasets(files[0], format_func=format_ds)

### Compute longitude / latitudes
dx = dy = 10000
lons, lats = compute_lonlat_from_regular_xy_grid(
    ds[LONGITUDE],
    ds[LATITUDE],
    dx=dx,
    dy=dy,
)
xs, ys  = lonlat_to_xy(lons, lats)
    
### Compute β-plane parameters

lat0 = (lats.max() + lats.min()) / 2
beta_plane = BetaPlane(
    f0=2 * EARTH_ANGULAR_ROTATION * np.sin(lat0),
    beta=2 * EARTH_ANGULAR_ROTATION * np.cos(lat0) / EARTH_RADIUS,
)
f0 = beta_plane.f0

### Build regridder

psi_regridder = build_regridder(ds, lons, lats)
sst_regridder = build_regridder(ds, interpolate(lons), interpolate(lats))

nx, ny = lats.shape
xx = torch.tensor(xs.round(), **specs)
space_2d = SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=xx - xx[0, :],
        y=torch.tensor(ys.round(), **specs),
    )
)

space_interior = space_2d.slice(
    b,
    space_2d.psi.xy.x.shape[0] - b,
    b,
    space_2d.psi.xy.x.shape[1] - b,
)

nx = space_interior.nx
ny = space_interior.ny
dx = space_interior.dx
dy = space_interior.dy

In [ ]:
from qgsw.configs.core import Configuration


ROOT_PATH = Path(os.getcwd()).parent

# Simulation parameters

config = Configuration.from_toml(ROOT_PATH.joinpath("config/enatl60.toml"))

H = config.model.h
H1,H2,H3 = H
g_prime = config.model.g_prime
g1,g2,g3= g_prime
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

y_w = space_2d.q.xy.y[0, :].unsqueeze(0)
y0 = 0.5 * space_interior.ly
beta_effect = beta_plane.beta * (y_w - y0)

In [ ]:
from qgsw.eNATL60.var_keys import SST
from qgsw.scripts.boundaries import extract_psi_bc, extract_sst_bc
from qgsw.solver.boundary_conditions.base import Boundaries
from typing import Literal, overload

from qgsw.eNATL60.fields_computations import compute_streamfunction_with_atmospheric_pressure_xy_avg
from qgsw.eNATL60.var_keys import ATMOS_PRESSURE, MERIDIONAL_WIND_10M, SSH, STREAMFUNCTION, TIME, ZONAL_WIND_10M
from qgsw.scripts.eNATL60 import da_to_tensor, load_netcdfs
from qgsw.utils.reshaping import crop

files_for_cycle = files[0:5]


@overload
def load_cycle_data(*files:Path, c:int,load_wind:Literal[True]) -> tuple[xr.DataArray,torch.Tensor,torch.Tensor,torch.Tensor, list[Boundaries], torch.Tensor]:...
@overload
def load_cycle_data(*files:Path, c:int,load_wind:Literal[False]) -> tuple[xr.DataArray,torch.Tensor,torch.Tensor, torch.Tensor, list[Boundaries]]:...

def load_cycle_data(*files:Path, c:int,load_wind:bool) -> tuple[xr.DataArray,torch.Tensor,torch.Tensor,list[Boundaries]]|tuple[xr.DataArray,torch.Tensor,torch.Tensor,list[Boundaries],torch.Tensor]:

    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        return

    files_for_cycle = files[start_cycle:end_cycle]

    with load_netcdfs(
        files_for_cycle, data_folder / "misc", load_wind=load_wind
    ) as ds:
        ds[STREAMFUNCTION] = (
            compute_streamfunction_with_atmospheric_pressure_xy_avg(
                ds[SSH],
                ds[ATMOS_PRESSURE],
                config.physics.rho,
                g_prime[0].item(),
                remove_avgs=True,
            )
        )

        with logger.timeit("Interpolating dataset"):
            ds_interp: xr.Dataset = psi_regridder(
                ds[
                    [STREAMFUNCTION]
                    + [ZONAL_WIND_10M, MERIDIONAL_WIND_10M] * load_wind
                ],
                output_chunks=(-1, -1),
            )
            ds_interp[LONGITUDE] = (["i", "j"], lons)
            ds_interp[LATITUDE] = (["i", "j"], lats)

        with logger.timeit("Interpolating SST"):
            regridded_sst: xr.DataArray = sst_regridder(
                ds[SST], output_chunks=(-1, -1)
            )
            ds_sst_interp = xr.Dataset(
                {
                    LONGITUDE: (["i", "j"], interpolate(lons)),
                    LATITUDE: (["i", "j"], interpolate(lats)),
                    SST: regridded_sst,
                },
                regridded_sst.coords,
            )
            ds_sst_interp = ds_sst_interp.set_coords([LONGITUDE, LATITUDE])

    with logger.timeit("Building tensors"):
        psis = da_to_tensor(ds_interp[STREAMFUNCTION], **specs) / f0
        ssts = da_to_tensor(ds_sst_interp[SST], **specs) + 273.15
        t0 = ds_interp[TIME][0]
        times = (ds_interp[TIME] - t0).dt.total_seconds().to_numpy()
        times = torch.tensor(times, **specs)
    with logger.timeit("Retrieving boundaries"):
        sst_bcs = [extract_sst_bc(s, b) for s in ssts]

    if load_wind:
        u10 = ds_interp[ZONAL_WIND_10M].to_numpy()
        v10 = ds_interp[MERIDIONAL_WIND_10M].to_numpy()
        uv10 = torch.stack(
            [
                crop(torch.tensor(u10, **specs), b),
                crop(torch.tensor(v10, **specs), b),
            ],
        )
        return ds[STREAMFUNCTION], times, psis, ssts, sst_bcs, uv10
    return ds[STREAMFUNCTION], times, psis, ssts, sst_bcs  

In [ ]:
def xy_to_lonlat(x:np.ndarray, y:np.ndarray, lat0:float) -> tuple[np.ndarray, np.ndarray]:
    lats = (y/EARTH_RADIUS)+lat0
    lons = x/EARTH_RADIUS/np.cos(lats)
    return lons,lats

def format_dsu(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "sozocrtx": "u",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])
def format_dsv(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "somecrty": "v",
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

    
space_no_xoffset=SpaceDiscretization2D.from_psi_grid(
    Grid2D(
        x=torch.tensor(xs.round(),**specs),
        y=torch.tensor(ys.round(), **specs),
    )
)

ux,uy  =space_no_xoffset.u.xy
ux=ux.cpu().numpy()
uy=uy.cpu().numpy()

lonsu, latsu = xy_to_lonlat(ux,uy,lat0)

vx,vy  =space_no_xoffset.v.xy
vx=vx.cpu().numpy()
vy=vy.cpu().numpy()

lonsv, latsv = xy_to_lonlat(vx,vy,lat0)

ufiles = list(
    (data_folder/"MEANDERS"/"gridU").glob("*.nc"),
)
ufiles = sort_files_by_dates(*ufiles)

dsu_init = load_datasets(ufiles[0], format_func=format_dsu)

u_regridder = build_regridder(dsu_init, lonsu, latsu)

vfiles = list(
    (data_folder/"MEANDERS"/"gridV").glob("*.nc"),
)
vfiles = sort_files_by_dates(*vfiles)

dsv_init = load_datasets(vfiles[0], format_func=format_dsv)

v_regridder = build_regridder(dsv_init, lonsv, latsv)

def load_cycle_uv_data(ufiles:np.ndarray, vfiles:np.ndarray,c:int) -> tuple[list[torch.Tensor],list[torch.Tensor], list[torch.Tensor]]:

    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(ufiles):
        msg = f"Not enough ufiles to perform cycle {c} and above."
        logger.warning(msg)
        return

    ufiles_for_cycle = ufiles[start_cycle:end_cycle]

    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(vfiles):
        msg = f"Not enough vfiles to perform cycle {c} and above."
        logger.warning(msg)
        return

    vfiles_for_cycle = vfiles[start_cycle:end_cycle]

    dsu = load_datasets(*ufiles_for_cycle, format_func=format_dsu)

    with logger.timeit("Interpolating u"):
        regridded_u: xr.DataArray = u_regridder(dsu["u"])
        dsu_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsu),
                LATITUDE: (["i", "j"], latsu),
                "u": ([TIME, "i", "j"], regridded_u.data),
            },
            regridded_u.coords,
        )
        dsu_interp = dsu_interp.set_coords([LONGITUDE, LATITUDE])
        dsu_interp = dsu_interp.load()

    dsv = load_datasets(*vfiles_for_cycle, format_func=format_dsv)

    with logger.timeit("Interpolating v"):
        regridded_v: xr.DataArray = v_regridder(dsv["v"])
        dsv_interp = xr.Dataset(
            {
                LONGITUDE: (["i", "j"], lonsv),
                LATITUDE: (["i", "j"], latsv),
                "v": ([TIME, "i", "j"], regridded_v.data),
            },
            regridded_v.coords,
        )
        dsv_interp = dsv_interp.set_coords([LONGITUDE, LATITUDE])
        dsv_interp = dsv_interp.load()
    

    u_refs = [
        torch.tensor(u, **specs).unsqueeze(0).unsqueeze(0)
        for u in dsu_interp["u"].to_numpy()
    ]
    v_refs = [
        torch.tensor(v, **specs).unsqueeze(0).unsqueeze(0)
        for v in dsv_interp["v"].to_numpy()
    ]

    return u_refs, v_refs

In [ ]:
from qgsw.eNATL60.var_keys import VORTICITY


data_folder = get_path_from_env(key="eNATL60_FOLDER")
vort_files = list(
    (data_folder/"MEANDERS"/"gridVort").glob("*.nc"),
)
vort_files = sort_files_by_dates(*vort_files)

def format_dvort(ds: xr.Dataset) -> xr.Dataset:
    """Format Dataset."""
    # Drop useless variables
    if "axis_nbounds" in ds.dims:
        ds = ds.drop_dims("axis_nbounds")
    if "time_centered" in ds.coords:
        ds = ds.reset_coords("time_centered", drop=True)
    # Rename
    ds = ds.rename(
        {
            "time_counter": TIME,
            "nav_lon": LONGITUDE,
            "nav_lat": LATITUDE,
            "x": "i",
            "y": "j",
            "vorticity": VORTICITY,
        }
    )
    ds = ds.transpose(TIME, "i", "j")
    return ds.set_coords([LONGITUDE, LATITUDE])

def load_cycle_vorticity_data(files:np.ndarray, c:int) -> torch.Tensor:
    start_cycle = c * n_file_per_cycle + c * separation
    end_cycle = (c + 1) * n_file_per_cycle + c * separation

    if end_cycle > len(files):
        msg = f"Not enough files to perform cycle {c} and above."
        logger.warning(msg)
        return

    files_for_cycle = files[start_cycle:end_cycle]

    with load_datasets(*files_for_cycle, format_func=format_dvort) as dvort:

        ds_lon = dvort[LONGITUDE]
        ds_lat=dvort[LATITUDE]

        lon_slice = (ds_lon >= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lon <= np.rad2deg(lons[(b+1):-(b+1),(b+1):-(b+1)].max()))
        lat_slice = (ds_lat >= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].min())) & (ds_lat <= np.rad2deg(lats[(b+1):-(b+1),(b+1):-(b+1)].max()))

        Is,Js = np.meshgrid(dvort.i.values,dvort.j.values,indexing="ij")

        I_slice = Is[lon_slice&lat_slice]
        J_slice = Js[lon_slice&lat_slice]

        imin,imax = I_slice.min(),I_slice.max()
        jmin,jmax = J_slice.min(),J_slice.max()

        dvort = dvort.sel(i=slice(imin,imax+1),j=slice(jmin,jmax+1))

        vorticities = torch.tensor(dvort[VORTICITY].to_numpy(), **specs).unsqueeze(0).unsqueeze(0)/beta_plane.f0
    return vorticities

In [ ]:
from qgsw.metrics.spectral import MagnitudeSquaredCoherence, PowerSpectralDensity
from qgsw.solver.finite_diff import grad_perp


def rmse(f: torch.Tensor, f_ref: torch.Tensor) -> float:
    """RMSE."""
    return ((f - f_ref).square().mean() / f_ref.square().mean()).sqrt()

def uv_rmse(psi:torch.Tensor,u_ref:torch.Tensor, v_ref:torch.Tensor) -> torch.Tensor:
    """Gradient RMSE."""
    u,v = grad_perp(psi)
    
    u/=dy
    v/=dx


    return ((u-u_ref).square().mean()+(v-v_ref).square().mean()).sqrt() / (u_ref.square().mean()+v_ref.square().mean()).sqrt()

psd_ = PowerSpectralDensity(
    nx=space_interior.psi.nx,
    ny=space_interior.psi.ny,
    dx=space_interior.psi.dx.item(),
    dy=space_interior.psi.dy.item(),
)

def psd_err(f:torch.Tensor, f_ref:torch.Tensor) -> torch.Tensor:
    return (psd_.iso_spec_w(f-f_ref)/psd_.iso_spec_w(f_ref))


msc_ = MagnitudeSquaredCoherence(
    nx=space_interior.psi.nx,
    ny=space_interior.psi.ny,
    dx=space_interior.psi.dx.item(),
    dy=space_interior.psi.dy.item(),
)


def msc(f:torch.Tensor, f_ref:torch.Tensor)->torch.Tensor:
    return msc_.iso_spec_w(f, f_ref)

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import TypeVar

from matplotlib import pyplot as plt
import numpy as np
from torch import Tensor
from qgsw.eNATL60.wind import compute_windstress
from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.masks import Masks
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.spatial.core.discretization import SpaceDiscretization2D

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T]):
    results_paths = Path("../output/g5k/param_optim")

    losses:dict[str,list[list[torch.Tensor]]] = {}
    uv10_to_uvsurf = torch.eye(2,**specs)
    
    _dt_default = dt

    show=True
    remember_psis = False
    no_wind=True

    @property
    def dt(self) -> int:
        return self._dt
    @dt.setter
    def dt(self, value:int) -> None:
        if (v:= data_dt // value)*value - data_dt != 0:
            raise ValueError(f"Data dt {data_dt} is not compatible with model dt {value}")
        self._substep_ratio = v
        self._dt=value
        self._set_params()
        self.model.dt=value

    @property
    def substep_ratio(self) -> int:
        """Substep ratio."""
        return self._substep_ratio
        

    def __init__(self, space_2d: SpaceDiscretization2D) -> None:
        super().__init__(space_2d)
        self.losses:dict[str,list[list[torch.Tensor]]] = {
            "rmse": [],
            "uv_rmse": [],
            "psd_err":[],
            "msc":[],
            "sst_rmse": []
        }
    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts: torch.Tensor, sst_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

    def _set_params(self) -> None:
        space = self.model.space
        self.model.masks = Masks.empty_tensor(
            space.nx,
            space.ny,
            device=specs["device"],
        )
        self.model.bottom_drag_coef = 0
        self.model.wide = True
        self.model.slip_coef = slip_coef
        
    def load(self)-> dict:
        file = self.results_paths.joinpath(self.prefix+".pt")
        if isinstance(r:=torch.load(file),dict):
            return r["results"]
        return r
    def new_cycle(self) -> None:
        super().new_cycle()
        for loss in self.losses.values():
            loss.append([])
        if self.remember_psis:
            self.psis = []
            
    def add_loss(self, loss_value:torch.Tensor,loss_name:str) -> None:
        self.losses[loss_name][-1].append(loss_value)
    def add_sst_loss(self, sst_ref:torch.Tensor,loss_fn:Callable[[torch.Tensor,torch.Tensor], torch.Tensor], loss_name:str) -> None:
        try:
            loss_value = loss_fn(self.model.sst[0,0],sst_ref)
        except AttributeError:
            loss_value = torch.tensor(np.nan, **specs)
        self.losses[loss_name][-1].append(loss_value)
        
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        if not self.show:
            return
        if ax is None:
            ax = plt.gca()
        cycles = [cycle] if cycle is not None else list(range(len(self.losses[loss_name])))
        time_offset= 0
        for i,c in enumerate(cycles):
            times = data_dt*np.arange(len(self.losses[loss_name][c]))/3600/24 + time_offset
            time_offset = times[-1] + data_dt/3600/24
            loss =  torch.stack(self.losses[loss_name][c]).cpu().numpy()
            kwargs = self.plot_kwargs.copy()
            if i!= 0:
                kwargs.pop("label")
            ax.plot(times, loss, **kwargs)
        ax.set_xlabel("Time [day]")
    
    def compute_windstress(self, uv10:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return compute_windstress(uv10,self.uv10_to_uvsurf,rho_water=config.physics.rho,rho_air=1.255)
    
    def step(self) -> None:
        for _ in range(self.substep_ratio):
            self.model.step()
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

        
            
        
M = TypeVar("M", bound=ModelWrapperOBC[QGPSIQCore])

class ModelsManagerOBC(ModelsManager[M]):

    loss_fn: dict[str, Callable[[torch.Tensor,torch.Tensor], torch.Tensor]]= {
        "rmse":rmse,
        "psd_err" : psd_err,
        "msc" : msc,
        "sst_rmse": rmse,
    }

    losses = list(loss_fn.keys())

    def remember_psis(self, mode:bool=True)-> None:
        for mw in self.model_wrappers:
            mw.remember_psis = mode

    def compute_loss(self, psi_ref:torch.Tensor) -> None:
        for loss_name in self.losses:
            self.loop_over_models(
                lambda mw: mw.add_loss(self.loss_fn[loss_name](mw.model.psi[0,0],psi_ref[0,0]),loss_name)
            )

    def compute_sst_loss(self, sst_ref:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.add_sst_loss(sst_ref[0,0],self.loss_fn["sst_rmse"], "sst_rmse")
        )

    def compute_uv_losses(self,u_ref:torch.Tensor, v_ref:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.add_loss(uv_rmse(mw.model.psi[0,0],u_ref[0,0],v_ref[0,0]),"uv_rmse")
        )
    
    def plot_loss(self,*,loss_name:str,ax:plt.Axes|None=None,cycle:int|None=None) -> None:
        self.loop_over_models(lambda mw: mw.plot_loss(loss_name=loss_name,ax=ax,cycle=cycle))
    
    def build_wind_forcing(self, uv10:torch.Tensor) -> None:
        self.loop_over_models(
            lambda mw: mw.set_wind_forcing(*mw.compute_windstress(uv10))
        )

    def setup(
        self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, sst_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        """Setup all models."""
        self.loop_over_models(lambda mw: mw.setup(sf_da,times,ssts,sst_bcs,beta_effect_w))

### Reduced Gravity

In [ ]:
from qgsw.models.qg.psiq.core import QGPSIQ
from qgsw.models.qg.psiq.mixed_layer.core import QGPSIQSSTAdv
from qgsw.scripts.boundaries import extract_psi_bc, extract_q_bc
from qgsw.scripts.eNATL60 import filter_streamfunction
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.spatial.core.grid_conversion import interpolate
from qgsw.utils.interpolation import QuadraticInterpolation



class ReducedGravity(ModelWrapperOBC[QGPSIQSSTAdv]):
    prefix = None
    color = "black"
    label="Reduced Gravity"
    sigma_ic = 16
    sigma_bc = 16
    no_wind=False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQSSTAdv(
            space_2d=space_2d,
            H=H[:1],
            beta_plane=beta_plane,
            g_prime=g_prime[:1]*g_prime[1:2]/(g_prime[:1]+g_prime[1:2]),
        )
        self.dt = data_dt
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(laplacian(psi,dx,dy) - beta_plane.f0**2 * (1/H1/g1+1/H1/g2)*psi[...,1:-1,1:-1]) + beta_effect
    
    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, ssts_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        psi0_filt_da, psis_filt_da = filter_streamfunction(
            sf_da,
            self.sigma_ic,
            self.sigma_bc,
        )
        psi0_filt_da: xr.DataArray = psi_regridder(
            psi0_filt_da, output_chunks=(-1, -1)
        )
        da_interp: xr.Dataset = psi_regridder(
            psis_filt_da,
            output_chunks=(-1, -1),
        )
        psi0 = da_to_tensor(psi0_filt_da, **specs) / beta_plane.f0
        psis_f = da_to_tensor(da_interp, **specs) / beta_plane.f0
        psi_bcs = [extract_psi_bc(psi[:,:1],b,clone=False) for psi in psis_f]
        q_bcs = [extract_q_bc(self.compute_q(psi[:, :1],beta_effect_w),b,clone=False)for psi in psis_f]
        del psis_f
        self.model.set_psiqsst(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1),crop(ssts[0],b))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs), QuadraticInterpolation(times, ssts_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

### RGSI SST

In [ ]:
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.models.qg.psiq.mixed_layer.forced import QGPSIQSSTRGSI
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde
from qgsw.utils.tensor_dict import change_specs


class RGSISST(ModelWrapperOBC[QGPSIQSSTRGSI]):
    prefix = "results_enatl60"
    color = "sandybrown"
    label = "RGSI - SST"
    save_video = False
    no_wind=False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQSSTRGSI(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, ssts_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cycle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        
        try:
            sigma_ic = res[self.cycle]["config"]["sigma_ic"]
            sigma_bc = res[self.cycle]["config"]["sigma_bc"]
        except KeyError:
            sigma_ic = 14
            sigma_bc = 14
        try:
            self.dt = res[self.cycle]["config"]["dt"]
        except KeyError:
            self.dt = data_dt
        try:
            self.model.H_ml = res[self.cycle]["H_ml"]
        except KeyError:...
        try:
            self.model.temp_1_offset = res[self.cycle]["temp_1_offset"]
        except KeyError:...

        psi0_filt_da, psis_filt_da = filter_streamfunction(
            sf_da,
            sigma_ic,
            sigma_bc,
        )
        psi0_filt_da: xr.DataArray = psi_regridder(
            psi0_filt_da, output_chunks=(-1, -1)
        )
        da_interp: xr.Dataset = psi_regridder(
            psis_filt_da,
            output_chunks=(-1, -1),
        )
        psi0 = da_to_tensor(psi0_filt_da, **specs) / beta_plane.f0
        psis_f = da_to_tensor(da_interp, **specs) / beta_plane.f0
        psi_bcs = [extract_psi_bc(psi[:,:1],b,clone=False) for psi in psis_f]
        q_bcs = [extract_q_bc(self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w),b,clone=False)for psi in psis_f]
        del psis_f
        self.model.set_psiqsst(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1),crop(ssts[0],b))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs), QuadraticInterpolation(times, ssts_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

In [ ]:
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.models.qg.psiq.mixed_layer.forced import QGPSIQSSTAdvRGSI
from qgsw.models.qg.psiq.modified.forced import QGPSIQRGPsi2TransportDR
from qgsw.models.qg.stretching_matrix import compute_A_tilde
from qgsw.utils.tensor_dict import change_specs


class RGSISSTAdv(ModelWrapperOBC[QGPSIQSSTAdvRGSI]):
    prefix = "results_enatl60"
    color = "lightcoral"
    label = "RGSI - SST advected"
    save_video = False
    no_wind=False
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQSSTAdvRGSI(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, ssts_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cycle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        
        try:
            sigma_ic = res[self.cycle]["config"]["sigma_ic"]
            sigma_bc = res[self.cycle]["config"]["sigma_bc"]
        except KeyError:
            sigma_ic = 14
            sigma_bc = 14
        try:
            self.dt = res[self.cycle]["config"]["dt"]
        except KeyError:
            self.dt = data_dt

        psi0_filt_da, psis_filt_da = filter_streamfunction(
            sf_da,
            sigma_ic,
            sigma_bc,
        )
        psi0_filt_da: xr.DataArray = psi_regridder(
            psi0_filt_da, output_chunks=(-1, -1)
        )
        da_interp: xr.Dataset = psi_regridder(
            psis_filt_da,
            output_chunks=(-1, -1),
        )
        psi0 = da_to_tensor(psi0_filt_da, **specs) / beta_plane.f0
        psis_f = da_to_tensor(da_interp, **specs) / beta_plane.f0
        psi_bcs = [extract_psi_bc(psi[:,:1],b,clone=False) for psi in psis_f]
        q_bcs = [extract_q_bc(self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w),b,clone=False)for psi in psis_f]
        del psis_f
        self.model.set_psiqsst(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1),crop(ssts[0],b))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs), QuadraticInterpolation(times, ssts_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

### RGSI

In [ ]:
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.models.qg.psiq.mixed_layer.forced import QGPSIQSSTRGSI
from qgsw.models.qg.stretching_matrix import compute_A_tilde
from qgsw.utils.tensor_dict import change_specs


class RGSI(ModelWrapperOBC[QGPSIQRGPsi2TransportDR]):
    prefix = "results_enatl60"
    color = "navy"
    label = "RGSI"
    save_video = False
    no_wind=True
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQRGPsi2TransportDR(
            space_2d=space_2d,
            H=H[:2],
            beta_plane=beta_plane,
            g_prime=g_prime[:2],
        )
        self.alphas = {}
        self.coefs = {}
    def compute_q(self,psi: Tensor, A11:torch.Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi, dx, dy)
            - beta_plane.f0**2 * A11 * psi[..., 1:-1, 1:-1]
        ) + beta_effect
    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, ssts_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        res = self.load()

        try:
            alpha:torch.Tensor = res[self.cycle]["alpha"]
        except KeyError:
            alpha=torch.tensor(0,**specs)
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        self.A = compute_A_tilde(H[:2],g_prime[:2],alpha,**specs)
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))

        basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        try:
            basis.freeze_time_normalization(self.model.dt*torch.tensor([n_steps_per_cycle],**specs))
        except:... 
        basis.set_coefs(coefs)
        self._fpsi2 = basis.localize(
            space_interior.psi.xy.x,space_interior.psi.xy.y
        )

        if self.save_params:
            self.alphas[self.cycle] = alpha
            self.coefs[self.cycle] = coefs
        
        try:
            sigma_ic = res[self.cycle]["config"]["sigma_ic"]
            sigma_bc = res[self.cycle]["config"]["sigma_bc"]
        except KeyError:
            sigma_ic = 14
            sigma_bc = 14
        try:
            self.dt = res[self.cycle]["config"]["dt"]
        except KeyError:
            self.dt = data_dt

        psi0_filt_da, psis_filt_da = filter_streamfunction(
            sf_da,
            sigma_ic,
            sigma_bc,
        )
        psi0_filt_da: xr.DataArray = psi_regridder(
            psi0_filt_da, output_chunks=(-1, -1)
        )
        da_interp: xr.Dataset = psi_regridder(
            psis_filt_da,
            output_chunks=(-1, -1),
        )
        psi0 = da_to_tensor(psi0_filt_da, **specs) / beta_plane.f0
        psis_f = da_to_tensor(da_interp, **specs) / beta_plane.f0
        psi_bcs = [extract_psi_bc(psi[:,:1],b,clone=False) for psi in psis_f]
        q_bcs = [extract_q_bc(self.compute_q(psi[:, :1],self.A[:1,:1],beta_effect_w),b,clone=False)for psi in psis_f]
        del psis_f
        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],self.A[:1,:1],beta_effect_w),b-1))
        self.model.alpha = alpha
        self.model.basis = basis
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

### Forced

In [ ]:
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced

Heq = H[:1]*H[1:2]/(H[:1]+H[1:2])
class Forced(ModelWrapperOBC[QGPSIQForced]):
    prefix = "results_enatl60_forced"
    color="brown"
    label="Forced DR"
    save_video = False
    no_wind=True
    def _init_model(self, space_2d:SpaceDiscretization2D) -> None:
        self.model= QGPSIQForced(
            space_2d=space_2d,
            H=Heq,
            beta_plane=beta_plane,
            g_prime=g_prime[1:2],
        )
        self.coefs = {}
    def _set_params(self) -> None:
        super()._set_params()
        self.model.wind_scaling = H[:1].item()
        
    def compute_q(self,psi: Tensor, beta_effect:torch.Tensor) -> Tensor:
        return interpolate(
            laplacian(psi,dx,dy)
            - beta_plane.f0**2 * (1/Heq/g2)*psi[...,1:-1,1:-1]
        ) + beta_effect

    def setup(self,sf_da:xr.DataArray, times:list[torch.Tensor], ssts:torch.Tensor, ssts_bcs:list[Boundaries],beta_effect_w:torch.Tensor) -> None:
        res = self.load()
        try:
            self.uv10_to_uvsurf = res[self.cycle]["uv10_to_uvsurf"]
        except KeyError:
            ...
        coefs = DecompositionCoefs.from_dict(change_specs(res[self.cycle]["coefs"],**specs))
        self.basis = build_basis_from_params_dict(res[self.cycle]["config"]["basis"])
        self.basis.set_coefs(coefs)
        if self.save_params:
            self.coefs[self.cycle] = coefs
            
        self.wv = self.basis.localize(
            self.model.space.remove_h().q.xy.x,
            self.model.space.remove_h().q.xy.y,
        )
        try:
            sigma_ic = res[self.cycle]["config"]["sigma_ic"]
            sigma_bc = res[self.cycle]["config"]["sigma_bc"]
        except KeyError:
            sigma_ic = 14
            sigma_bc = 14
        try:
            self.dt = res[self.cycle]["config"]["dt"]
        except KeyError:
            self.dt = data_dt
        psi0_filt_da, psis_filt_da = filter_streamfunction(
            sf_da,
            sigma_ic,
            sigma_bc,
        )
        psi0_filt_da: xr.DataArray = psi_regridder(
            psi0_filt_da, output_chunks=(-1, -1)
        )
        da_interp: xr.Dataset = psi_regridder(
            psis_filt_da,
            output_chunks=(-1, -1),
        )
        psi0 = da_to_tensor(psi0_filt_da, **specs) / beta_plane.f0
        psis_f = da_to_tensor(da_interp, **specs) / beta_plane.f0
        psi_bcs = [extract_psi_bc(psi[:,:1],b,clone=False) for psi in psis_f]
        q_bcs = [extract_q_bc(self.compute_q(psi[:, :1],beta_effect_w),b,clone=False)for psi in psis_f]
        del psis_f

        self.model.set_psiq(crop(psi0[:,:1],b), crop(self.compute_q(psi0[:,:1],beta_effect_w),b-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())
        
    def step(self) -> None:
        self.model.forcing = self.wv(self.model.time)
        super().step()

In [ ]:
def perform_cycle(season:str,c:int, models:ModelsManagerOBC, n_steps:int=n_steps_per_cycle,comparison_interval:int=1) -> list[torch.Tensor]:
    in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
    if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
        msg = "Non-time-contiguous data for this season in provided dataset."
        raise ValueError(msg)
    season_files = files[in_season]
    season_ufiles = ufiles[in_season]
    season_vfiles = vfiles[in_season]

    models.new_cycle()

    sf_da, times, psis_ref, ssts_ref, ssts_bcs, uv10=load_cycle_data(*season_files,c=c,load_wind=True)

    u_refs, v_refs = load_cycle_uv_data(season_ufiles, season_vfiles,c)

    models.setup(sf_da.compute(), times, ssts_ref, ssts_bcs, beta_effect[:,1:-1])

    models.reset_time()

    models.compute_loss(crop(psis_ref[0],b))
    models.compute_uv_losses(crop(u_refs[0],b),crop(v_refs[0],b))
    for n in range(1,n_steps-1):
        models.build_wind_forcing(uv10[:,n-1])
        models.step()
        if n % comparison_interval == 0:
            models.compute_loss(crop(psis_ref[int(n//comparison_interval)],b))
            models.compute_sst_loss(crop(ssts_ref[int(n//comparison_interval)],b))
            models.compute_uv_losses(crop(u_refs[int(n//comparison_interval)],b),crop(v_refs[int(n//comparison_interval)],b))
    return psis_ref, ssts_ref


In [ ]:
import gc

In [ ]:
from qgsw.logging.utils import box, step

season = "summer"

rg = ReducedGravity(space_interior)
rg.dt = 3600
rg.sigma_bc = 10
rg.sigma_ic = 10

forced = Forced(space_interior)
forced.prefix = f"results_enatl60_forced_atmp_hr_gamma1000_obstrack_{season}_re2"
forced.linestyle = "solid"
forced.label = "RG-F"

rgsi_sst = RGSISST(space_interior)
rgsi_sst.results_paths = Path("../output/local/param_optim/test_sst")
rgsi_sst.prefix = f"sst_gamma0_01_gammasst0_01_obstrack_o100_{season}"
rgsi_sst.linestyle = "solid"
rgsi_sst.label = "RGSI - SST"

rgsi_sst_noconv = RGSISST(space_interior)
rgsi_sst_noconv.results_paths = Path("../output/local/param_optim/test_sst")
rgsi_sst_noconv.prefix = f"sst_noconv_gamma0_01_gammasst0_01_obstrack_o100_{season}"
rgsi_sst_noconv.linestyle = "dotted"
rgsi_sst_noconv.label = "RGSI - SST no conv"
rgsi_sst_noconv.model.with_no_convection()

rgsi_sst_adv = RGSISSTAdv(space_interior)
rgsi_sst_adv.results_paths = Path("../output/local/param_optim/test_sst")
rgsi_sst_adv.prefix = f"sst_adv_gamma0_01_gammasst0_01_obstrack_o100_{season}"
rgsi_sst_adv.linestyle = "solid"
rgsi_sst_adv.label = "RGSI - SST advected"

rgsi = RGSI(space_interior)
rgsi.prefix = f"results_enatl60_atmp_hr_gamma0_001_obstrack_{season}"
rgsi.linestyle = "solid"
rgsi.label = "RGSI"


in_season = retrieve_dates(*files.tolist()).month.isin(season_dict[season])
if ((in_season[1:]) & (~in_season[:-1])).sum() + int(in_season[0]) > 1:
    msg = "Non-time-contiguous data for this season in provided dataset."
    raise ValueError(msg)
season_vort_files = vort_files[in_season]

models = ModelsManagerOBC(
    # rg, 
    # forced, 
    rgsi_sst_adv,
    rgsi_sst, 
    rgsi_sst_noconv,
    # rgsi
)
models.save_params = True

models.remember_psis(False)

c_start = 0
c_end = 1

for _ in range(0, c_start):
    models.new_cycle()

gc.collect()
for c in range(c_start, c_end):#n_cycles):

    times = np.arange(n_steps_per_cycle)*dt+c*n_steps_per_cycle*dt

    torch.cuda.reset_peak_memory_stats()
    psis_ref, ssts_ref = perform_cycle(season,c,models)

    # vorticities = load_cycle_vorticity_data(season_vort_files, c)

    # make_vorticity_video(f"../output/videos/eNATL60/{season}/vorticity_atmp_c{c}.mp4", vorticities, times/3600/24, rg, forced, rgsi_g0_001,forcedsurf, robust=True)

    # make_psi_video(f"../output/videos/eNATL60/{season}/psi_atmp_c{c}.mp4",psis_ref,times/3600/24,rg,forced,rgsi_g0_001,forcedsurf)

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="="))


In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show
# forced_g10000.linestyle = "solid"
# Plots

show_grad = True

fig,axs = plots.subplots(1+show_grad,1,figsize=(22,(5)*(1+show_grad)))
fig.suptitle(season.capitalize())
plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0])
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[1,0])
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

ref = crop(psis_ref[-2][0,0],b)

vmax = ref.abs().max()

plots.imshow(ref,ax=axs[0,0],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)==0)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(mw.model.psi[0,0],ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

ref = laplacian(psis_ref[-2][0,0],dx,dy)

vmax = ref.abs().quantile(0.99)

plots.imshow(ref,ax=axs[0,0],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)==0)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(mw.model.vorticity[0,0],ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

ref = crop(ssts_ref[-2][0,0],b)

vmin, vmax = ref.min(),ref.max()

plots.imshow(ref,ax=axs[0,0],vmin=vmin,vmax=vmax,show_cbar=len(models.model_wrappers)==0,cmap="magma")

for i,mw in enumerate(models.model_wrappers):
    try:
        plots.imshow(mw.model.sst[0,0],ax=axs[0,i+1],vmin=vmin,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i,cmap="magma")
    except AttributeError:...
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,2)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(msc_.kr.cpu(),torch.tensor(mw.losses["msc"][0][-1]).cpu(),label = mw.label)
    axs[0,1].plot(msc_.kr.cpu(),torch.stack([torch.tensor(l) for l in mw.losses["msc"][0]]).mean(axis=0).cpu(),label = mw.label)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,2)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(psd_.kr.cpu(),torch.tensor(mw.losses["psd_err"][0][-1]).cpu(),label = mw.label)
    axs[0,1].plot(psd_.kr.cpu(),torch.stack([torch.tensor(l) for l in mw.losses["psd_err"][0]]).mean(axis=0).cpu(),label = mw.label)

plt.legend()
plt.tight_layout()
plt.show()